In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("../data/cleaned_transit_delays.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Baseline Model: Predicting the average delay for each route
route_avg_delay = df.groupby('route_id')['delay_minutes'].transform('mean')
baseline_mae = np.mean(np.abs(df['delay_minutes'] - route_avg_delay))
print(f"Baseline MAE: {baseline_mae}")

df['route_id_encoded'] = df['route_id'].astype('category').cat.codes
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

# Random Forest Model
features = ['route_id_encoded', 'stop_id', 'hour', 'day_of_week']
X = df[features]
y = df['delay_minutes']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f" Random Forest Model MAE: {mae}")

important_features = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print(important_features)



Baseline MAE: 5.36459708276174
 Random Forest Model MAE: 4.463571502877282
stop_id             0.522273
route_id_encoded    0.230496
hour                0.137619
day_of_week         0.109613
dtype: float64
